# 對等關稅對台汽車零售業的衝擊分析

本專案旨在分析總體經濟指標（如匯率、經濟成長率、人均所得）與台灣汽車銷售量（國產車 vs. 進口車）之間的關聯性，並建立預測模型（ARIMA, Prophet, LightGBM）來評估對等關稅政策實施後對車市造成的衝擊。

In [ ]:
# 1. 匯入必要的套件
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import missingno as msno
from pmdarima import auto_arima
from prophet import Prophet
import lightgbm as lgb
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_absolute_error
import plotly.graph_objects as go
import warnings

warnings.filterwarnings('ignore') # 隱藏不必要的警告

## 資料讀取與前處理
將經濟數據與汽車銷量合併，並處理缺失值與時間格式。

In [ ]:
# 讀取資料 (請確保 data 資料夾與本檔案位於同一個目錄下)
df_econ = pd.read_csv('./data/2014-2025 Economic Indicators.csv')
df_car = pd.read_csv('./data/2014-2025 Quarterly car sales.csv')

# 合併資料
df_total = pd.concat([df_econ, df_car[['Domestic_vehicles', 'Imported_vehicles']]], axis=1)

# 重新命名與計算總銷量
df_rename = df_total.rename(columns={'Domestic_vehicles':'Domestic_sales', 'Imported_vehicles':'Imported_sales'})
df_rename['Total_Sales'] = df_rename['Domestic_sales'] + df_rename['Imported_sales']

# 移除缺失值 (最新一期經濟數據無法推估，故直接移除)
df_total_new = df_rename.dropna()

# 處理時間索引
df_total_new['Statistical_Period'] = pd.PeriodIndex(df_total_new['Statistical_Period'], freq='Q')
df_total_new.set_index('Statistical_Period', inplace=True)

# 建立季度欄位
df_total_new['quarter'] = df_total_new.index.quarter
df_total_new['quarter_str'] = 'Q' + df_total_new['quarter'].astype(str)

# 移除不需要的欄位
df_total_new.drop(['Mid_term_Population (persons)', 'GDP', 'National_Income'], axis=1, inplace=True)
df_total_new.head()

## 探索性資料分析 (EDA)
透過熱力圖、箱型圖與時間序列圖，觀察各項變數之間的關聯性。

In [ ]:
# 繪製熱力圖
plt.figure(figsize=(10, 8))
sns.heatmap(df_total_new.corr(numeric_only=True), annot=True, square=False, cmap='Blues')
plt.title('Correlation Heatmap')
plt.show()

# 各季度平均銷量統計與箱型圖
plt.figure(figsize=(12, 8), layout='constrained')
sns.boxplot(x='quarter_str', y='Total_Sales', data=df_total_new, palette='viridis')
plt.title('Box Plot of Total Sales Distribution by Quarter', fontsize=20)
plt.xlabel('Quarter', fontsize=14)
plt.ylabel('Total Sales', fontsize=14)
plt.show()

# 時間序列圖：汽車銷量 vs 經濟成長率
fig, ax1 = plt.subplots(figsize=(18, 8), layout='constrained')
ax1.plot(df_total_new.index.to_timestamp(), df_total_new['Domestic_sales'], color='royalblue', label='Domestic_sales')
ax1.plot(df_total_new.index.to_timestamp(), df_total_new['Imported_sales'], color='darkorange', label='Imported_sales')
ax1.plot(df_total_new.index.to_timestamp(), df_total_new['Total_Sales'], color='tab:purple', label='Total_Sales')
ax1.set_xlabel('Year', fontsize=14)
ax1.set_ylabel('Quarterly sales (units)', color='black', fontsize=14)
ax1.legend(loc='upper left', fontsize=12)

ax2 = ax1.twinx()
ax2.plot(df_total_new.index.to_timestamp(), df_total_new['Economic_Growth_Rate'], color='tab:green', label='Economic_Growth_Rate')
ax2.set_ylabel('Economic_Growth_Rate', color='tab:green', fontsize=14)
ax2.legend(loc='upper right', fontsize=12)

plt.title("Taiwan's vehicle sales time series trend (2014-2025) VS Economic_Growth_Rate", fontsize=20)
plt.grid(True, which='both', linestyle='--', linewidth=0.5)
plt.show()

## 模型預測：ARIMA, Prophet, LightGBM
切分訓練集與測試集 (以 2025Q2 為界)，並訓練三個不同的時間序列模型來預測總銷量。

In [ ]:
# 切分資料
df_train = df_total_new[df_total_new.index < '2025Q2']
df_test = df_total_new[df_total_new.index >= '2025Q2']

features = ['Exchange_Rate', 'Income_Per_Capita', 'Economic_Growth_Rate']
X_train, y_train = df_train[features], df_train['Total_Sales']
X_test, y_test = df_test[features], df_test['Total_Sales']

# === 1. ARIMA ===
arima_model = auto_arima(y_train, exogenous=X_train, trace=False, error_action='ignore', suppress_warnings=True, seasonal=True, m=4)
arima_pred = arima_model.predict(n_periods=len(X_test), exogenous=X_test)
arima_gap = y_test - arima_pred

# === 2. Prophet ===
fbp_model = Prophet(seasonality_mode='multiplicative')
for feature in features:
    fbp_model.add_regressor(feature)

fbp_train = df_train.reset_index().rename(columns={'Statistical_Period': 'ds', 'Total_Sales': 'y'})
fbp_test = df_test.reset_index().rename(columns={'Statistical_Period': 'ds', 'Total_Sales': 'y'})
fbp_train['ds'] = fbp_train['ds'].dt.to_timestamp(how='end')
fbp_test['ds'] = fbp_test['ds'].dt.to_timestamp(how='end')

fbp_model.fit(fbp_train[['ds','y'] + features])
fbp_pred = fbp_model.predict(fbp_test[['ds'] + features])
fbp_gap = pd.merge(fbp_test[['ds', 'y']], fbp_pred[['ds', 'yhat']], on='ds')
fbp_gap['gap'] = fbp_gap['y'] - fbp_gap['yhat']

# === 3. LightGBM ===
lgbm_features = features + ['quarter']
param_grid = {'n_estimators': [100, 200], 'learning_rate': [0.05, 0.1], 'max_depth': [3, 5]}
lgbm = lgb.LGBMRegressor(random_state=42)
grid_search = GridSearchCV(estimator=lgbm, param_grid=param_grid, scoring='neg_mean_absolute_error', cv=3, verbose=0, n_jobs=-1)
grid_search.fit(df_train[lgbm_features], y_train)
lgbmr_pred = grid_search.best_estimator_.predict(df_test[lgbm_features])
lgbm_gap = y_test - lgbmr_pred

## 最終綜合分析與共識結論
結合三種模型的預測結果，計算平均預測值與缺口，並視覺化綜合比較。

In [ ]:
# 繪製測試集的綜合比較圖
fig_compare = go.Figure()
fig_compare.add_trace(go.Bar(x=df_test.index.to_timestamp(how='end'), y=y_test, name='實際銷量', marker_color='black'))
fig_compare.add_trace(go.Bar(x=df_test.index.to_timestamp(how='end'), y=arima_pred, name='ARIMA預測'))
fig_compare.add_trace(go.Bar(x=df_test.index.to_timestamp(how='end'), y=fbp_pred['yhat'], name='Prophet預測'))
fig_compare.add_trace(go.Bar(x=df_test.index.to_timestamp(how='end'), y=lgbmr_pred, name='LGBM預測'))
fig_compare.update_layout(title='綜合比較圖', xaxis_title='季度', yaxis_title='總銷量(輛)', barmode='group')
fig_compare.show()

# 整理綜合結論 DataFrame
fbp_pred_series = fbp_pred['yhat'].copy()
fbp_pred_series.index = y_test.index
fbp_gap_series = (y_test - fbp_pred_series)

avg_sales_forecast = (arima_pred + lgbmr_pred + fbp_pred_series) / 3
avg_gap_series = (arima_gap + fbp_gap_series + lgbm_gap) / 3
avg_gap_total_mean = avg_gap_series.mean()

final_summary_df = pd.DataFrame({
    '實際銷量': y_test,
    'ARIMA預測': arima_pred,
    'Prophet預測': fbp_pred_series,
    'LGBM預測': lgbmr_pred,
    '共識銷量 (Avg)': avg_sales_forecast,
    '共識缺口 (Gap)': avg_gap_series
})

print('\n' + '='*80)
print('                        最終綜合分析總表')
print('='*80)
print(final_summary_df.round(0).to_markdown(index=True))
print('\n' + '='*80)
print('                        最終綜合結論')
print('='*80)
print(f"本研究整合了 ARIMA、Prophet 及 LightGBM 三種模型的預測結果... 依據三模型的平均預估值可以得知，市場總銷售量每季約減少 {abs(int(avg_gap_total_mean))} 輛。")
print('='*80)